Setting up github connection

In [ ]:

!git clone "https://github.com/OwenHHHH/stat-359-Final-Project"
%cd stat-359-Final-Project

# setting github identity and access
TOKEN = "Private Access Token"
USER = "OwenHHHH"
REPO = "stat-359-Final-Project"
!git config --global user.email "owenhandelman@gmail.com"
!git config --global user.name "OwenHHHH"
!git remote set-url origin https://{TOKEN}@github.com/{USER}/{REPO}.git

Creating the base models

In [ ]:
import json
import os

for num_layers in [2, 6, 4, 8]:
    for num_heads in [2, 4, 8, 16]:
      initial_config = {
          "vocab_size": 1000, # based on tokenizer
          "d_model": 256, # everything other than layers and heads is left standard
          "nhead": num_heads,
          "num_layers": num_layers,
          "dim_feedforward": 1024,
          "max_seq_length": 512,
          "dropout": 0.1
      }

      with open('model_config.json', 'w') as f:
          json.dump(initial_config, f)

        # using the given run_foundational_training.py to create the base models and saving the output
      !python -m Given_Files.run_foundational_training \
      --corpus-path Data/foundational_corpus.txt \
      --tokenizer-path Data \
      --model-config model_config.json \
      --num-epochs 3 \
      --batch-size 16 \
      --device cuda \
      --output-dir Models/layer{num_layers}_head{num_heads}_base

      !git add -f Models/layer{num_layers}_head{num_heads}_base/**/best_model.pt
      !git add -f Models/layer{num_layers}_head{num_heads}_base/**/*.json
      !git commit -m "base model l{num_layers} h{num_heads}"
      !git push origin main

LoRA Training the Base Models

In [ ]:
for layer_num in [2, 4, 6, 8]:
  for head_num in [2, 4, 8, 16]:
    for lora_rank in [4, 8, 16, 32]:
      # using the given file run_instruction_training_lora.py for LoRA training - keeping everything as given except varying rank and setting epochs to 1
      !python -m Given_Files.run_instruction_training_lora \
      --instruction-corpus-path Data/instruction_corpus.txt \
      --tokenizer-path Data \
      --foundational-checkpoint Models/layer{layer_num}_head{head_num}_base/**/best_model.pt \
      --lora-rank {lora_rank} \
      --num-epochs 1 \
      --batch-size 32 \
      --device cuda \
      --output-dir Models/layer{layer_num}_head{head_num}_lora_rank{lora_rank}
        # saving the output in github
      !git add -f Models/layer{layer_num}_head{head_num}_lora_rank{lora_rank}/**/lora_adapter.pt
      !git add -f Models/layer{layer_num}_head{head_num}_lora_rank{lora_rank}/**/*.json
      !git commit -m "lora l{layer_num} h{head_num} lora{lora_rank}"
      !git push origin main

Evaluating all of the models

In [ ]:
import glob

for layer_num in [2, 4, 6, 8]:
  for head_num in [2, 4, 8, 16]:
    for lora_rank in [4, 8, 16, 32]:
      # selecting the most recently trained model
      model_matches = glob.glob(f"Models/layer{layer_num}_head{head_num}_lora_rank{lora_rank}/**/lora_adapter.pt", recursive=True)
      base_matches = glob.glob(f"Models/layer{layer_num}_head{head_num}_base/**/best_model.pt", recursive=True)
      if model_matches and base_matches:
          model_path = model_matches[-1]
          base_path = base_matches[-1]
      else:
        continue
        # using the base given run_evaluation.py
      !python -m Given_Files.run_evaluation \
      --model-path {model_path} \
      --base-checkpoint {base_path} \
      --tokenizer-path Data \
      --max-depth 5 \
      --num-range 0 99 \
      --num-samples 500 \
      --batch-size 32 \
      --output-dir Results/layer{layer_num}_head{head_num}_lora_rank{lora_rank}_max_depth5

      !git add -f Results/layer{layer_num}_head{head_num}_lora_rank{lora_rank}_max_depth5
        # using the edited version called strict_depth_run_evaluation which only tests one depth (only on depth 1, depth 2, etc)
      for depth in [1, 2, 3, 4, 5]:
        !python -m Given_Files.strict_depth_run_evaluation \
        --model-path {model_path} \
        --base-checkpoint {base_path} \
        --tokenizer-path Data \
        --max-depth {depth} \
        --num-range 0 99 \
        --num-samples 100 \
        --batch-size 32 \
        --output-dir Results/layer{layer_num}_head{head_num}_lora_rank{lora_rank}_depth{depth}

        !git add -f Results/layer{layer_num}_head{head_num}_lora_rank{lora_rank}_depth{depth}
        # adding all of the evaluations to git hub
      !git commit -m "Full evaluation l{layer_num} h{head_num} lora_rank{lora_rank}, across all depths individually and collectively"
      !git push origin main

Graphs and Analysis

In [ ]:
import glob
import json
import os
import pandas as pd
from datetime import datetime

def parse_ablation_results(results_path="Results"):
    all_metrics = []
    all_samples = []

    # get all eval metric files and create dictionary to store information
    metric_files = glob.glob(f"{results_path}/**/*evaluation_metrics_*.json", recursive=True)
    latest_files = {}

    for filepath in metric_files:
        filename = os.path.basename(filepath)
        parts = filepath.split('/')
        folder_name = parts[1]

        # get model data from folder name
        try:
            config_parts = folder_name.split('_')
            l = config_parts[0].replace('layer', '')
            h = config_parts[1].replace('head', '')
            r = config_parts[3].replace('rank', '')
            depth_type = config_parts[4] 
            timestamp_str = "_".join(filename.split('_')[-2:]).replace('.json', '')
            ts = datetime.strptime(timestamp_str, "%Y%m%d_%H%M%S")

            config_key = (l, h, r, depth_type)
            # only keeping the most recent evaluation
            if config_key not in latest_files or ts > latest_files[config_key][0]:
                latest_files[config_key] = (ts, filepath)
        except:
            continue

    # get the metrics from each file
    for (l, h, r, d_type), (ts, path) in latest_files.items():
        with open(path, 'r') as f:
            data = json.load(f)
            metric_entry = {
                "Layers": int(l),
                "Heads": int(h),
                "Rank": int(r),
                "Test_Type": d_type,
                "Accuracy": data['exact_match_accuracy'],
                "Parse_Rate": data['parse_success_rate'],
                "Avg_Gen_Len": data['avg_generation_length']
            }
            all_metrics.append(metric_entry)

    # get all sample output files
    sample_files = glob.glob(f"{results_path}/**/*sample_outputs_*.json", recursive=True)
    latest_files = {}

    for filepath in sample_files:
        filename = os.path.basename(filepath)
        parts = filepath.split('/')
        folder_name = parts[1]

        # get model data from folder name
        try:
            config_parts = folder_name.split('_')
            l = config_parts[0].replace('layer', '')
            h = config_parts[1].replace('head', '')
            r = config_parts[3].replace('rank', '')
            depth_type = config_parts[4]
            timestamp_str = "_".join(filename.split('_')[-2:]).replace('.json', '')
            ts = datetime.strptime(timestamp_str, "%Y%m%d_%H%M%S")

            config_key = (l, h, r, depth_type)
            # only keeping the most recent evaluation's outputs
            if config_key not in latest_files or ts > latest_files[config_key][0]:
                latest_files[config_key] = (ts, filepath)
        except:
            continue

    # get all of the sample output data
    for (l, h, r, d_type), (ts, path) in latest_files.items():
        with open(path, 'r') as f:
                samples = json.load(f)
                for s in samples:
                    all_samples.append({
                        "Config": f"L{l}H{h}R{r}",
                        "Depth": d_type,
                        "Input": s.get('expression'),
                        "Target": s.get('ground_truth'),
                        "Pred": s.get('predicted_answer'),
                        "Correct": s.get('is_correct'),
                        "Reasoning": s.get('model_output', '')[:100] + "..."
                    })

    return pd.DataFrame(all_metrics), pd.DataFrame(all_samples)

df_metrics, df_samples = parse_ablation_results()

accuracy_bydepth = df_metrics.pivot_table(
    index=['Layers', 'Heads', 'Rank'],
    columns='Test_Type',
    values='Accuracy'
).reset_index()

parse_success_bydepth = df_metrics.pivot_table(
    index=['Layers', 'Heads', 'Rank'],
    columns='Test_Type',
    values='Parse_Rate'
).reset_index()

In [ ]:
# saving the data
import os
summary_dir = "Summary_Data"
os.makedirs(summary_dir, exist_ok=True)

df_metrics.to_csv(f"{summary_dir}/full_metrics_table.csv", index=False)
accuracy_bydepth.to_csv(f"{summary_dir}/depth_vs_architecture_pivot.csv", index=False)
df_samples.to_csv(f"{summary_dir}/model_output_samples.csv", index=False)
parse_success_bydepth.to_csv(f"{summary_dir}/depth_vs_architecture_parse_pivot.csv", index=False)

!git add Summary_Data/*.csv
!git commit -m "Add consolidated results and pivot tables"
!git push origin main

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# showing digit token length
tokens_df = pd.read_csv('Data/token_table.csv')
plt.figure(figsize=(10, 6))
tokens_df['len'] = tokens_df['Token'].apply(lambda x: len(str(x)))
tokens_df_filter = tokens_df[tokens_df['Type'] == 'Digit']
ax = sns.countplot(x = 'len', data = tokens_df_filter)
ax.bar_label(ax.containers[0])
plt.xlabel('Digit Token Length')
plt.ylabel('Count')
plt.title("Tokenizer Vocabulary: Digit Token Length Distribution")
plt.savefig("Summary_Data/tokenizer_lengths.png", dpi=300)
plt.show()

In [ ]:
df_acc = pd.read_csv('Summary_Data/depth_vs_architecture_pivot.csv')
df_parse = pd.read_csv('Summary_Data/depth_vs_architecture_parse_pivot.csv')
depth_labels = ['depth1', 'depth2', 'depth3', 'depth4', 'depth5', 'max']
# plotting how accuracy changes across depth holding heads at 8 and LoRA Rank at 16
plt.figure()
df_acc_r16 = df_acc[df_acc['Rank'] == 16]
df_acc_h8_r16 = df_acc_r16[df_acc_r16['Heads'] == 8]
for layers in [2, 4, 6, 8]:
  subset = df_acc_h8_r16[df_acc_h8_r16['Layers'] == layers][['depth1', 'depth2', 'depth3', 'depth4', 'depth5']]
  plt.plot(['1', '2', '3', '4', '5'], subset.iloc[0], marker = 'o', label = f'{layers} Layers')
plt.title('Accuracy Decay by Depth and Layer (8 Attention Heads, LoRA Rank 16)')
plt.ylabel('Exact Match Accuracy %')
plt.xlabel('Depth')
plt.legend()
plt.savefig('Summary_Data/accuracy_decay_by_depth.png', dpi=300)
plt.show()

# plotting 
df_acc_l6_r16 = df_acc_r16[df_acc_r16['Layers'] == 6]
for heads in [2, 4, 8, 16]:
  subset = df_acc_l6_r16[df_acc_l6_r16['Heads'] == heads][['depth1', 'depth2','depth3', 'depth4', 'depth5']]
  plt.plot(['1', '2', '3', '4', '5'], subset.iloc[0], marker = 'o', label = f'{heads} Heads')
plt.title('Accuracy Decay by Depth and Attention Heads (6 Layers, LoRA Rank 16)')
plt.ylabel('Exact Match Accuracy %')
plt.xlabel('Depth')
plt.legend()
plt.savefig('Summary_Data/accuracy_decay_by_heads_and_depth.png', dpi=300)
plt.show()

# plotting how parse rate changes across attention heads holding layers at 6 and LoRA Rank at 16
plt.figure()
df_parse_r16 = df_parse[df_parse['Rank'] == 16]
df_parse_l6_r16 = df_parse_r16[df_parse_r16['Layers'] == 6]
for heads in [2, 4, 8, 16]:
  subset = df_parse_l6_r16[df_parse_l6_r16['Heads'] == heads][['depth1', 'depth2', 'depth3', 'depth4', 'depth5']]
  plt.plot(['1', '2', '3', '4', '5'], subset.iloc[0], marker = 'o', label = f'{heads} Heads')
plt.title('Parse Success by Attention Heads (6 Layers, LoRA Rank 16)')
plt.ylabel('Parse Success Rate %')
plt.xlabel('Depth')
plt.legend()
plt.savefig('Summary_Data/parse_success_by_depth.png', dpi=300)
plt.show()

# plotting parse success rate for a model with 6 layers and 8 heads as depth increasing (each LoRA rank is a different line)
plt.figure()
df_parse_l6 = df_parse[df_parse['Layers'] == 6]
df_parse_l6_h8 = df_parse_l6[df_parse_l6['Heads'] == 8]
for ranks in [4, 8, 16, 32]:
  subset = df_parse_l6_h8[df_parse_l6_h8['Rank'] == ranks][['depth1', 'depth2', 'depth3', 'depth4', 'depth5']]
  plt.plot(['1', '2', '3', '4', '5'], subset.iloc[0], marker = 'o', label = f'LoRA Rank {ranks}')
plt.title('Parse Success by LoRA Rank (6 Layers, 8 Attention Heads)')
plt.ylabel('Parse Success Rate %')
plt.xlabel('Depth')
plt.legend()
plt.savefig('Summary_Data/parse_success_by_lora_rank.png', dpi=300)
plt.show()

In [ ]:
# heatmap showing parse success rate by layers and attention heads and LoRa rank kept at 16
r16_max_parse_table = df_parse_r16.groupby(['Layers', 'Heads'])['max'].mean().unstack()
r16_max_parse_table = r16_max_parse_table / 100
plt.figure()
sns.heatmap(r16_max_parse_table, annot=True, cmap="YlGnBu", fmt = '.1%')
plt.title('Parse Success Rate: Attention Heads vs Layers (LoRA Rank 16)')
plt.savefig('Summary_Data/parse_by_attention_and_layers.png')
plt.show()

# heatmap showing parse success rate by attention heads and LoRA Rank (layers kept at 6)
df_parse_l6 = df_parse[df_parse['Layers'] == 6]
l6_max_parse_table = df_parse_l6.groupby(['Heads', 'Rank'])['max'].mean().unstack()
l6_max_parse_table = l6_max_parse_table / 100
plt.figure()
sns.heatmap(l6_max_parse_table, annot = True, cmap="YlGnBu", fmt = '.1%')
plt.title('Parse Success Rate: Attention Heads vs LoRA Rank (6 Layers)')
plt.savefig('Summary_Data/parse_by_attention_and_rank.png')
plt.show()

In [ ]:
# saving figures to github
!git add Summary_Data
!git commit -m "Adding figures"
!git push origin main